# 📋 Notebook 01 — Data Preparation (Dual Dataset)
## VERA: Visual Evidence–Report Alignment

This notebook prepares our dataset from two sources on Kaggle:
1. **Indiana University Chest X-rays** (mounted at `/kaggle/input/chest-xrays-indiana-university/`)
2. **ReXGradient-160K** (loaded from HuggingFace via streaming)

**Steps:**
1. Load Indiana U samples (XML parsing)
2. Load ReXGradient samples (streaming subset)
3. Combine and create train/val/test splits (70/20/10)
4. Sanity check: display images and reports from both sources

## 1. Imports & Config

In [ ]:
!pip install -q datasets Pillow

import sys
import os
from pathlib import Path

# Ensure project root is in path
if os.path.exists('/kaggle/working'):
    PROJECT_ROOT = Path('/kaggle/working')
else:
    PROJECT_ROOT = Path(os.getcwd()).parent if 'notebooks' in os.getcwd() else Path(os.getcwd())
sys.path.insert(0, str(PROJECT_ROOT))

from config import (
    IU_DATA_DIR, OUTPUT_DIR, REX_SUBSET_SIZE, HF_DATASET_ID, HF_TOKEN
)
from src.data_utils import (
    load_indiana_u, load_rexgradient, combine_datasets
)

import matplotlib.pyplot as plt
from PIL import Image

print("✅ Imports successful. Config loaded:")
print(f"  IU Data Dir: {IU_DATA_DIR}")
print(f"  Output Dir:  {OUTPUT_DIR}")
print(f"  ReX Subset:  {REX_SUBSET_SIZE} samples")

## 2. Load Indiana University Dataset

In [ ]:
print("Loading Indiana University Dataset...")
iu_samples = load_indiana_u()

print(f"\n✅ Loaded {len(iu_samples)} Indiana U samples.")
if iu_samples:
    sample = iu_samples[0]
    print("\n--- Sample IU Report ---")
    print(f"Image ID: {sample['image_id']}")
    print(f"Path: {sample['image_path']}")
    print(f"Report: {sample['report'][:200]}...")

## 3. Load ReXGradient Dataset

In [ ]:
print(f"Loading {REX_SUBSET_SIZE} ReXGradient samples via streaming...")
rex_samples = load_rexgradient()

print(f"\n✅ Loaded {len(rex_samples)} ReXGradient samples.")
if rex_samples:
    sample = rex_samples[0]
    print("\n--- Sample ReX Report ---")
    print(f"Image ID: {sample['image_id']}")
    print(f"Path: {sample['image_path']}")
    print(f"Report: {sample['report'][:200]}...")

## 4. Combine Datasets and Create Splits

In [ ]:
print("Merging datasets and generating 70/20/10 splits...")
splits = combine_datasets(iu_samples, rex_samples)

print("\n✅ Split generated and saved:")
for split_name, split_data in splits.items():
    print(f"  {split_name.capitalize()}: {len(split_data)} samples")

## 5. Sanity Check: Display Examples

In [ ]:
# Find one sample from each source in the train split
train_data = splits['train']
iu_example = next((s for s in train_data if s['source'] == 'indiana_u'), None)
rex_example = next((s for s in train_data if s['source'] == 'rexgradient'), None)

examples = []
if iu_example: examples.append(('Indiana U', iu_example))
if rex_example: examples.append(('ReXGradient', rex_example))

if examples:
    fig, axes = plt.subplots(1, len(examples), figsize=(12, 6))
    if len(examples) == 1:
        axes = [axes]
        
    for i, (source_name, sample) in enumerate(examples):
        img = Image.open(sample['image_path']).convert('L')
        axes[i].imshow(img, cmap='gray')
        
        report_snippet = sample['report'][:150] + '...'
        title = f"{source_name}\nID: {sample['image_id']}\n\n{report_snippet}"
        
        axes[i].set_title(title, fontsize=10, wrap=True)
        axes[i].axis('off')
        
    plt.tight_layout()
    plt.show()
else:
    print("No samples found to display.")